# Task 3 - GRU (Kaggle/Colab)
Train GRU for the three required areas and save predictions + metrics.

In [ ]:
# Setup and data load
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras

DATA_PATH = Path("/kaggle/input/telecom-traffic/city_traffic.parquet")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Parquet not found at {DATA_PATH}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
df = df.sort_values(["square_id", "time_interval"])
df["time_interval"] = pd.to_datetime(df["time_interval"], utc=True)

def get_area_series(data: pd.DataFrame, square_id: int) -> pd.Series:
    s = data[data["square_id"] == square_id].sort_values("time_interval")
    return s.set_index("time_interval")["internet_traffic"]

def split_train_test(series: pd.Series):
    train_end = pd.Timestamp("2013-12-15 23:50:00", tz="UTC")
    test_start = pd.Timestamp("2013-12-16 00:00:00", tz="UTC")
    test_end = pd.Timestamp("2013-12-22 23:50:00", tz="UTC")
    train = series.loc[:train_end]
    test = series.loc[test_start:test_end]
    return train, test

def make_sequences(values: np.ndarray, seq_len: int):
    x, y = [], []
    for i in range(len(values) - seq_len):
        x.append(values[i : i + seq_len])
        y.append(values[i + seq_len])
    return np.array(x), np.array(y)

target_squares = [df.groupby("square_id")["internet_traffic"].sum().idxmax(), 4159, 4556]
SEQ_LEN = 144

In [ ]:
# GRU training + evaluation
def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

gru_results = []
for square_id in target_squares:
    series = get_area_series(df, int(square_id))
    train, test = split_train_test(series)

    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.values.reshape(-1, 1)).squeeze()
    test_scaled = scaler.transform(test.values.reshape(-1, 1)).squeeze()

    x_train, y_train = make_sequences(train_scaled, SEQ_LEN)
    x_test, y_test = make_sequences(np.concatenate([train_scaled[-SEQ_LEN:], test_scaled]), SEQ_LEN)
    x_train = x_train[..., np.newaxis]
    x_test = x_test[..., np.newaxis]

    model = keras.Sequential([
        keras.layers.Input(shape=(SEQ_LEN, 1)),
        keras.layers.GRU(64, return_sequences=False),
        keras.layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")

    start = time.perf_counter()
    model.fit(x_train, y_train, epochs=10, batch_size=128, verbose=1)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    pred_scaled = model.predict(x_test, verbose=0).squeeze()
    infer_time = time.perf_counter() - start

    y_pred = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).squeeze()
    y_true = test.values[SEQ_LEN:]
    metrics = evaluate_metrics(y_true, y_pred)
    metrics.update({"square_id": int(square_id), "train_seconds": train_time, "infer_seconds": infer_time})
    gru_results.append(metrics)

    out = pd.DataFrame({
        "time_interval": test.index[SEQ_LEN:],
        "y_true": y_true,
        "y_pred": y_pred,
        "model": "GRU",
        "square_id": int(square_id),
    })
    out.to_csv(OUTPUT_DIR / f"gru_{int(square_id)}.csv", index=False)

pd.DataFrame(gru_results).to_csv(OUTPUT_DIR / "metrics_gru.csv", index=False)
pd.DataFrame(gru_results)